# 9장 평가: 잘 됐는지 어떻게 아나 (실습)

교재 `docs/book/09-evaluation.md` 와 함께 본다. 이 노트북에서 하는 것:

1. `small-cpu` 의 val loss · perplexity · 글자당 bit, 1장 n-gram 부터 지금까지 한 표에
2. 모델 크기 vs 손실 (스케일링 맛보기): 크기 4종을 같은 스텝만 학습
3. 어휘 크기 비교: BPE 4,096 vs 8,192, 토큰당 loss 는 비교 불가, 글자당 bit 로

> 전체 실행 약 6분 (§2 가 대부분).

In [ ]:
import math
import time

import torch

from shllm.bigram import NGramModel
from shllm.config import TOKENIZER_DIR, setup_cpu
from shllm.data import load_corpus
from shllm.eval import bits_per_char, evaluate_loss, perplexity, scaling_experiment
from shllm.tokenizer import BPETokenizer, CharTokenizer
from shllm.train import load_checkpoint

setup_cpu()
text = load_corpus("korean-classics")
tok = BPETokenizer.load(TOKENIZER_DIR / "bpe-8192.json")
data = torch.tensor(tok.encode(text))
n = int(0.9 * len(data))
train, val = data[:n], data[n:]
val_chars = len(tok.decode(val.tolist()))
print(f"val: {len(val):,} 토큰 = {val_chars:,} 글자")

## 1. 지금까지의 모델을 한 표에

In [ ]:
rows = []
# 1장: 문자 n-gram (문자 토큰이라 글자당 환산이 곧 토큰당)
ctok = CharTokenizer.from_text(text)
cids = torch.tensor(ctok.encode(text))
cn = int(0.9 * len(cids))
for k in (2, 3):
    m = NGramModel(k, ctok.vocab_size).fit(cids[:cn])
    loss = m.loss(cids[cn:])
    rows.append((f"{k}-gram (문자, 세기)", loss, bits_per_char(loss, len(cids[cn:]), len(cids[cn:]))))
# BPE n-gram
for k in (2,):
    m = NGramModel(k, tok.vocab_size).fit(train)
    loss = m.loss(val)
    rows.append((f"{k}-gram (BPE, 세기)", loss, bits_per_char(loss, len(val), val_chars)))
# 7장 GPT: run 마다 자기 토크나이저로 **같은 val 텍스트**(근대문학 뒤 10%)를 인코딩해 평가 → 글자당 bit 로 비교 가능
val_text = tok.decode(val.tolist())
for run in ("tiny-notebook", "small-cpu", "mixed-cpu"):
    try:
        model, ck = load_checkpoint(run, "best.pt")
    except FileNotFoundError:
        continue
    rtok = BPETokenizer.load(TOKENIZER_DIR / ck.get("tokenizer", "bpe-8192.json"))
    rval = torch.tensor(rtok.encode(val_text))
    loss = evaluate_loss(model, rval, model.cfg.block_size, batch_size=16, n_batches=30)
    rows.append((f"GPT {run} ({model.n_params() / 1e6:.1f}M, step {ck['step']}, {ck.get('tokenizer', '')[:14]})", loss, bits_per_char(loss, len(rval), len(val_text))))

print(f"{'모델':<52}{'val loss/토큰':>14}{'perplexity':>12}{'bit/글자':>10}")
for name, loss, bpc in rows:
    print(f"{name:<52}{loss:>14.3f}{perplexity(loss):>12.1f}{bpc:>10.2f}")

**출력에서 볼 것**: 토큰당 loss 열은 문자 n-gram 이 가장 낮고, bit/글자 열은 GPT 가 가장 낮다. 두 열의 순서가 다르다는 것이 이 표의 핵심이다.

**해 보기**: `NGramModel(3, ...)` 대신 5-gram 을 넣어 bit/글자가 오르는지(1장 과적합) 확인하라. `n_batches=30` 을 100 으로 올리면 값이 얼마나 안정되나.

- **토큰당 loss 는 토크나이저가 다르면 비교할 수 없다.** 문자 2-gram 의 3.1 과 BPE 2-gram 의 7.7 은 "한 글자" 와 "두 글자 남짓" 을 맞히는 어려움의 차이다.
- **글자당 bit** 로 환산하면 같은 텍스트를 부호화하는 비용이라 공정하다. 세기 → MLP → 어텐션 → GPT 로 오면서 이 값이 내려온다.
- `mixed-cpu`(위키 혼합 코퍼스, 다른 토크나이저. ADR-0003)가 있으면 같은 근대문학 val 텍스트로 평가된 행이 추가된다. 토큰당 loss 는 토크나이저가 달라 비교 불가, **글자당 bit 만** 본다.
- perplexity 는 exp(loss). "매 순간 몇 개 중 하나를 찍는 셈인가". 사람이 읽는 데는 bit/글자가 더 직관적이다.

## 2. 모델 크기 vs 손실: 스케일링 맛보기

같은 데이터·같은 스텝(400)으로 크기만 바꿔 학습한다. 작은 실험이지만 "크게 만들면 좋아지는가"의 모양을 볼 수 있다.

In [ ]:
sizes = [
    {"n_layer": 1, "n_embd": 64, "n_head": 2},
    {"n_layer": 2, "n_embd": 128, "n_head": 4},
    {"n_layer": 4, "n_embd": 128, "n_head": 4},
    {"n_layer": 4, "n_embd": 256, "n_head": 8},
]
t0 = time.perf_counter()
results = scaling_experiment(train, val, sizes, steps=400, vocab_size=tok.vocab_size, block_size=64, batch_size=32)
print(f"총 {time.perf_counter() - t0:.0f}초")

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

cjk = [f.name for f in font_manager.fontManager.ttflist if "CJK" in f.name]
if cjk:
    matplotlib.rcParams["font.family"] = cjk[0]

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot([r["params"] for r in results], [r["val_loss"] for r in results], "o-")
for r in results:
    ax.annotate(f"L{r['n_layer']}·C{r['n_embd']}", (r["params"], r["val_loss"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set_xlabel("파라미터 수 (log)")
ax.set_ylabel("val loss (400 스텝)")
ax.set_title("같은 스텝 수에서 크기만 바꾸면")
plt.show()

대체로 파라미터가 늘수록 같은 스텝에서 loss 가 낮다. 큰 모델이 **더 빨리 배운다**. 스케일링 법칙(Kaplan 2020)이 말하는 것의 축소판이다. 예외가 하나 보인다: 같은 폭(C=128)에서 층만 2→4 로 늘린 모델은 400 스텝에서 오히려 조금 나쁘다. 깊은 모델은 초기에 더 느리게 출발하고, **폭**을 늘린 쪽(C=256)이 짧은 학습에서는 더 효율적이다.
단, 데이터가 고정이면 어느 크기부터는 과적합이 더 빨리 올 뿐 val 최저는 좋아지지 않는다(7장). 실제 스케일링은 **모델·데이터·계산을 함께** 키운다.

## 3. 어휘 크기: 4,096 vs 8,192

2장에서 미뤄 둔 결정. 같은 모델(2층·C=128)을 두 토크나이저로 같은 시간(400 스텝) 학습하고 **글자당 bit** 로 비교한다.

In [ ]:
big = BPETokenizer.load(TOKENIZER_DIR / "bpe-8192.json")
vocab_rows = []
for V in (4096, 8192):
    t = big.truncated(V)
    d = torch.tensor(t.encode(text))
    nn_ = int(0.9 * len(d))
    tr, va = d[:nn_], d[nn_:]
    chars = len(t.decode(va.tolist()))
    r = scaling_experiment(tr, va, [{"n_layer": 2, "n_embd": 128, "n_head": 4}], steps=400, vocab_size=V, block_size=64, batch_size=32, verbose=False)[0]
    vocab_rows.append((V, len(d), r["params"], r["val_loss"], bits_per_char(r["val_loss"], len(va), chars), r["seconds"]))
print(f"{'어휘':>6}{'코퍼스 토큰':>12}{'파라미터':>11}{'loss/토큰':>10}{'bit/글자':>9}{'초':>6}")
for V, ntok, p, loss, bpc, s in vocab_rows:
    print(f"{V:>6}{ntok:>12,}{p:>11,}{loss:>10.3f}{bpc:>9.2f}{s:>6.0f}")

토큰당 loss 만 보면 어휘가 작은 쪽이 항상 유리해 보인다(맞힐 후보가 적으니). 글자당 bit 로 보면 차이가 줄거나 뒤집힌다.
어휘가 크면 시퀀스가 짧아 같은 문맥 창(64 토큰)에 더 많은 글자가 들어가고 스텝당 더 많은 글자를 학습한다. 대신 임베딩이 커진다.
이 결과(셀 출력)를 근거로 프로젝트 어휘를 확정한다: 교재 9.4.

## 정리

- 비교의 단위를 맞춰라: 토큰당 loss 는 같은 토크나이저끼리만, 다르면 글자당 bit.
- perplexity = exp(loss). 균등 찍기(V) 에서 얼마나 내려왔는지의 감각.
- 크기를 키우면 빨리 배우지만 데이터가 고정이면 과적합이 빨리 온다. 모델·데이터·계산은 함께 키운다.

---
**다음 장**: 10장, 여기서 ChatGPT 까지: 파인튜닝·RLHF·RAG 의 지도, 그리고 작은 시연.